# TABLE Purge + Privilege Test (split out from the combined notebook)

## Why this notebook exists
The combined `table_view_purge_privilege_test.ipynb` was run live on 2026-07-06.
Real results exposed a finding that changes the whole approach for tables:

```
DROP TABLE ...?purgeRequested=true   (as ROOT)
  -> 403 "Principal 'root' with activated PrincipalRoles '[service_admin]' and
          activated grants via '[service_admin, catalog_admin]' is not
          authorized for op DROP_TABLE_WITH_PURGE"
```

**Root/`service_admin` does NOT bypass this check.** That contradicts the
assumption carried over from `view_purge_behavior_test.ipynb` (there, root
*does* bypass view-drop privilege checks -- proven live, C9). Tables have a
genuinely different authorization model: purging a table is gated by a
DISTINCT op (`DROP_TABLE_WITH_PURGE`), which requires `CATALOG_MANAGE_CONTENT`
to be **explicitly granted** on the catalog -- even for root. The live
non-root privilege probes (Part 4 of the combined notebook) already proved
this correctly:

```
table_baseline (no drop grant)          -> 403
table_with_table_drop (TABLE_DROP only) -> 403   <- contradicts doc-privilege-results.md
table_with_cmc (CATALOG_MANAGE_CONTENT) -> 204   <- this is what actually authorizes it
```

The combined notebook's smoking-gun and config-matrix cells never granted
`CATALOG_MANAGE_CONTENT` before attempting the purge drop (they assumed root
would bypass, as it does for views), so they hit this same 403 as an
**unexpected** result -- that's the bug being fixed here, not a Polaris bug.

## What changed here (table-only)
1. **`ensure_privilege()`** -- a pre-flight helper that reads a catalog-role's
   CURRENT grants and only issues a grant if the required privilege is
   missing, called BEFORE every drop-with-purge attempt where the test isn't
   deliberately probing privilege. This directly answers "check privilege
   before executing the command."
2. **The config-gate matrix now explicitly controls `purgeRequested`** for
   every case (previously several cases left it absent, which never engages
   `DROP_TABLE_WITH_PURGE` at all -- that's why they came back 204 instead of
   the previously-expected 403: the config gate was never actually being
   tested). Combined with (1), config-403 and privilege-403 are now
   distinguishable by message text, exactly like the original view notebook's
   "two different 403s" analysis, ported to tables.
3. Table-only -- no view cells. `view_purge_behavior_test.ipynb` and the
   combined `table_view_purge_privilege_test.ipynb` are untouched.

**Status:** static-verified only (mocked backend) -- this sandbox can't reach
the live OrbStack Polaris host. Run "Restart & Run All" against `local`.

In [1]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

# init_env() BEFORE `import *` -- see table_view_purge_privilege_test.ipynb's
# cell 1 note: init_env() sets BASE_MGMT/BASE_CAT/BUCKET/REALM/mc/etc. via
# `global` inside its OWN module namespace, so doing `import *` first would
# leave this notebook's copies of those names stale at their pre-init None
# defaults.
import polaris_test_utils as _ptu
_ptu.init_env("local")
from polaris_test_utils import *
require_not_prod("table purge + privilege matrix")

import uuid, time, json as _json
tok = root_token()

_CONFIG_KEYS = {
    "drop_with_purge_enabled": "polaris.config.drop-with-purge.enabled",
}

def set_catalog_config(catalog, _retries=5, **flags):
    want = {}
    for k, v in flags.items():
        if k not in _CONFIG_KEYS:
            raise KeyError(f"unknown config flag '{k}'. Known: {list(_CONFIG_KEYS)}")
        want[_CONFIG_KEYS[k]] = v
    for _ in range(_retries):
        cat = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json()
        merged = {**cat.get("properties", {}), **want}
        r = requests.put(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok),
                         json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
        if r.status_code in (200, 204):
            now = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json().get("properties", {})
            if all(now.get(k) == v for k, v in want.items()):
                return True
        elif r.status_code == 409:
            time.sleep(0.2); continue
        else:
            raise RuntimeError(f"set_catalog_config PUT failed [{r.status_code}]: {r.text[:200]}")
    raise RuntimeError(f"set_catalog_config could not apply {want} to {catalog}")

def get_catalog_config(cat):
    props = requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).json().get("properties", {})
    return {"drop-with-purge.enabled": props.get("polaris.config.drop-with-purge.enabled")}

DEBUG = False
print("setup ready (table-only)")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
setup ready (table-only)


In [2]:
# force_delete_catalog -- table-focused teardown (also sweeps any stray views
# defensively, but this notebook never creates any).
def _ns_name(ns):
    return ns[0] if isinstance(ns, list) else ns

def force_delete_catalog(cat):
    if requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code == 404:
        return True
    set_catalog_config(cat, drop_with_purge_enabled="true")
    grant_privilege(catalog=cat, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)

    nss = requests.get(f"{BASE_CAT}/{cat}/namespaces", headers=h(tok)).json().get("namespaces", [])
    nss = sorted(nss, key=lambda n: len(n if isinstance(n, list) else [n]), reverse=True)
    for ns in nss:
        nsn = _ns_name(ns)
        for t in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables/{t['name']}", headers=h(tok))
        for v in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views/{v['name']}", headers=h(tok))
        tleft = requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", [])
        vleft = requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", [])
        if tleft or vleft:
            print(f"   still has tables={len(tleft)} views={len(vleft)} in {nsn} -- cannot delete ns yet")
            continue
        requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}", headers=h(tok))

    for cr in requests.get(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles", headers=h(tok)).json().get("roles", []):
        crn = cr["name"] if isinstance(cr, dict) else cr
        if crn != "catalog_admin":
            requests.delete(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{crn}", headers=h(tok))
    rc = requests.delete(f"{BASE_MGMT}/catalogs/{cat}?purgeRequested=true", headers=h(tok))
    if mc is not None:
        mc.delete_prefix(f"{cat}/", min_size=0)
    still = requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code != 404
    if still:
        print(f"   catalog {cat} STILL EXISTS (delete status {rc.status_code}): {rc.text[:150]}")
        return False
    return True

def create_namespace_verified(cat, ns_name, _retries=4):
    """Create a namespace, tolerating BOTH the documented read-after-write
    500 (NPE, trust-the-commit) AND a transient 403 immediately after catalog
    creation -- the live run (2026-07-06, cell 15/C3) hit a 403 'not
    authorized for op CREATE...NAMESPACE' on a namespace create for ROOT right
    after a fresh catalog was created, which smells like the same PG-HA
    read-after-write lag documented elsewhere in this repo (the catalog's
    default catalog_admin bootstrap grant not yet visible). A short
    settle+retry is a defensive mitigation, not a confirmed root cause --
    flag to the platform owner if this keeps firing."""
    last = None
    for attempt in range(_retries):
        r = create_namespace(catalog=cat, ns=ns_name, token=tok)
        if r.status_code in (200, 201):
            return True
        if r.status_code == 500 and ("metadata" in r.text or "NullPointer" in r.text):
            time.sleep(0.6)
            return True
        if r.status_code == 403 and attempt < _retries - 1:
            time.sleep(0.5 * (attempt + 1))
            last = r
            continue
        raise RuntimeError(f"namespace {ns_name} create failed [{r.status_code}]: {r.text[:200]}")
    raise RuntimeError(f"namespace {ns_name} create failed after retries [{last.status_code}]: {last.text[:200]}")

print("force_delete_catalog + create_namespace_verified ready")

force_delete_catalog + create_namespace_verified ready


In [3]:
# THE FIX for the "unexpected 403": check privilege BEFORE executing the
# purge-drop command, and only grant if it's actually missing. Answers
# directly: "check privilege before execute command, whether correct
# privilege is requested."
#
# PROVEN LIVE (2026-07-06, non-root probes in the combined notebook):
#   TABLE_DROP alone             -> 403 (NOT sufficient -- contradicts
#                                    doc-privilege-results.md's claim)
#   CATALOG_MANAGE_CONTENT       -> 204 (sufficient)
# So the default required privilege for DROP_TABLE_WITH_PURGE is
# CATALOG_MANAGE_CONTENT, not TABLE_DROP.
REQUIRED_FOR_TABLE_PURGE = ["CATALOG_MANAGE_CONTENT"]

def current_grants(cat, cr):
    r = requests.get(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{cr}/grants", headers=h(tok))
    if r.status_code != 200:
        return set()
    return {g.get("privilege") for g in r.json().get("grants", [])}

def ensure_privilege(cat, cr, required, label=""):
    """Read CURRENT grants for (cat, cr); if none of `required` is already
    present, grant the first one. Prints what it found so the notebook's
    output makes the check visible, not just the fix."""
    have = current_grants(cat, cr)
    missing = not (have & set(required))
    tag = f"[{label}] " if label else ""
    if missing:
        to_grant = required[0]
        print(f"{tag}privilege check: {cr}@{cat} has {have or '{}'} -- missing {required} -> granting {to_grant}")
        grant_privilege(catalog=cat, cr=cr, privilege=to_grant, token=tok)
        return True
    print(f"{tag}privilege check: {cr}@{cat} already has {have & set(required)} -- ok, no grant needed")
    return False

print("ensure_privilege ready (pre-flight privilege check + grant-if-missing)")

ensure_privilege ready (pre-flight privilege check + grant-if-missing)


In [4]:
# Real Parquet + Iceberg v2 manifest/manifest-list + snapshot commit, written
# via MinioREST (src/minio_rest.py) -- no s3fs, per repo policy.
import pyarrow as pa
import pyarrow.parquet as pq
try:
    import fastavro
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "fastavro", "--quiet"])
    import fastavro
import io

_MANIFEST_SCHEMA = {
    "type": "record", "name": "manifest_entry",
    "fields": [
        {"name": "status",          "type": "int"},
        {"name": "snapshot_id",     "type": ["null","long"], "default": None},
        {"name": "sequence_number", "type": ["null","long"], "default": None},
        {"name": "file_sequence_number","type":["null","long"],"default":None},
        {"name": "data_file", "type": {
            "type": "record", "name": "r2",
            "fields": [
                {"name": "content",            "type": "int"},
                {"name": "file_path",          "type": "string"},
                {"name": "file_format",        "type": "string"},
                {"name": "partition",          "type": {"type":"record","name":"r102","fields":[]}},
                {"name": "record_count",       "type": "long"},
                {"name": "file_size_in_bytes", "type": "long"},
                {"name": "column_sizes",       "type":["null",{"type":"array","items":{"type":"record","name":"k81v81","fields":[{"name":"key","type":"int"},{"name":"value","type":"long"}]}}],"default":None},
                {"name": "value_counts",       "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "null_value_counts",  "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "nan_value_counts",   "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "lower_bounds",       "type":["null",{"type":"array","items":{"type":"record","name":"k81v100","fields":[{"name":"key","type":"int"},{"name":"value","type":"bytes"}]}}],"default":None},
                {"name": "upper_bounds",       "type":["null",{"type":"array","items":"k81v100"}],"default":None},
                {"name": "key_metadata",       "type":["null","bytes"],"default":None},
                {"name": "split_offsets",      "type":["null",{"type":"array","items":"long"}],"default":None},
                {"name": "equality_ids",       "type":["null",{"type":"array","items":"int"}],"default":None},
                {"name": "sort_order_id",      "type":["null","int"],"default":None},
            ]
        }}
    ]
}
_MANIFEST_LIST_SCHEMA = {
    "type": "record", "name": "manifest_file",
    "fields": [
        {"name": "manifest_path",              "type": "string"},
        {"name": "manifest_length",             "type": "long"},
        {"name": "partition_spec_id",           "type": "int"},
        {"name": "content",                     "type": "int",  "default": 0},
        {"name": "sequence_number",             "type": "long", "default": 0},
        {"name": "min_sequence_number",         "type": "long", "default": 0},
        {"name": "added_snapshot_id",           "type": "long"},
        {"name": "added_data_files_count",      "type": "int"},
        {"name": "existing_data_files_count",   "type": "int"},
        {"name": "deleted_data_files_count",    "type": "int"},
        {"name": "added_rows_count",            "type": "long"},
        {"name": "existing_rows_count",         "type": "long"},
        {"name": "deleted_rows_count",          "type": "long"},
        {"name": "partitions",  "type":["null",{"type":"array","items":{"type":"record","name":"r508","fields":[{"name":"contains_null","type":"boolean"},{"name":"contains_nan","type":["null","boolean"],"default":None},{"name":"lower_bound","type":["null","bytes"],"default":None},{"name":"upper_bound","type":["null","bytes"],"default":None}]}}],"default":None},
        {"name": "key_metadata","type":["null","bytes"],"default":None},
    ]
}

def write_real_table_data(cat, ns, table_name, minio_client=None):
    m = minio_client or mc
    base = f"{cat}/{ns}/{table_name}"
    table_data = pa.table({
        "id":   pa.array([1, 2, 3, 4, 5], type=pa.int64()),
        "name": pa.array(["alice", "bob", "carol", "dave", "eve"], type=pa.string()),
    })
    buf = io.BytesIO(); pq.write_table(table_data, buf)
    parquet_bytes = buf.getvalue()
    file_uuid   = uuid.uuid4().hex
    parquet_key = f"{base}/data/{file_uuid}.parquet"
    m.put_bytes(parquet_key, parquet_bytes)
    file_size = len(parquet_bytes)

    snapshot_id = int(time.time() * 1000)
    seq_num = 1
    manifest_entry = {
        "status": 1, "snapshot_id": snapshot_id,
        "sequence_number": seq_num, "file_sequence_number": seq_num,
        "data_file": {
            "content": 0, "file_path": f"s3a://{BUCKET}/{parquet_key}", "file_format": "PARQUET",
            "partition": {}, "record_count": 5, "file_size_in_bytes": file_size,
            "column_sizes": None, "value_counts": None, "null_value_counts": None,
            "nan_value_counts": None, "lower_bounds": None, "upper_bounds": None,
            "key_metadata": None, "split_offsets": None, "equality_ids": None, "sort_order_id": 0,
        }
    }
    manifest_key = f"{base}/metadata/{uuid.uuid4().hex}-m0.avro"
    mbuf = io.BytesIO()
    fastavro.writer(mbuf, fastavro.parse_schema(_MANIFEST_SCHEMA), [manifest_entry], metadata={
        "schema": _json.dumps({"type":"struct","schema-id":0,"fields":[
            {"id":1,"name":"id","required":True,"type":"long"},
            {"id":2,"name":"name","required":False,"type":"string"}]}),
        "partition-spec": "[]", "partition-spec-id": "0", "format-version": "2",
        "content": "data", "sequence-number": str(seq_num), "min-sequence-number": str(seq_num),
        "snapshot-id": str(snapshot_id), "added-data-files": "1", "existing-data-files": "0",
        "deleted-data-files": "0", "added-rows": "5", "existing-rows": "0", "deleted-rows": "0"})
    manifest_bytes = mbuf.getvalue()
    m.put_bytes(manifest_key, manifest_bytes)

    manifest_list_key = f"{base}/metadata/snap-{snapshot_id}-{uuid.uuid4().hex}.avro"
    mlbuf = io.BytesIO()
    fastavro.writer(mlbuf, fastavro.parse_schema(_MANIFEST_LIST_SCHEMA), [{
        "manifest_path": f"s3a://{BUCKET}/{manifest_key}", "manifest_length": len(manifest_bytes),
        "partition_spec_id": 0, "content": 0, "sequence_number": seq_num, "min_sequence_number": seq_num,
        "added_snapshot_id": snapshot_id, "added_data_files_count": 1, "existing_data_files_count": 0,
        "deleted_data_files_count": 0, "added_rows_count": 5, "existing_rows_count": 0, "deleted_rows_count": 0,
        "partitions": None, "key_metadata": None}])
    m.put_bytes(manifest_list_key, mlbuf.getvalue())

    r_commit = requests.post(
        f"{BASE_CAT}/{cat}/namespaces/{ns}/tables/{table_name}", headers=h(tok),
        json={"identifier": {"namespace": [ns], "name": table_name}, "requirements": [],
              "updates": [{"action": "add-snapshot", "snapshot": {
                  "snapshot-id": snapshot_id, "timestamp-ms": snapshot_id,
                  "summary": {"operation": "append", "added-data-files": "1", "added-records": "5",
                              "added-files-size": str(file_size), "total-data-files": "1", "total-records": "5"},
                  "schema-id": 0, "manifest-list": f"s3a://{BUCKET}/{manifest_list_key}"}}]})
    if r_commit.status_code in (200, 201, 204):
        requests.post(f"{BASE_CAT}/{cat}/namespaces/{ns}/tables/{table_name}", headers=h(tok),
            json={"identifier": {"namespace": [ns], "name": table_name}, "requirements": [],
                  "updates": [{"action": "set-snapshot-ref", "ref-name": "main",
                               "type": "branch", "snapshot-id": snapshot_id}]})
    else:
        print(f"  WARNING: snapshot commit failed [{r_commit.status_code}]: {r_commit.text[:200]}")

    return snapshot_id, [parquet_key, manifest_key, manifest_list_key], r_commit.status_code

print("write_real_table_data ready")

write_real_table_data ready


## Part 1 — Smoking gun: does `purgeRequested=true` remove files? (fixed)

Same as the combined notebook's Part 1, but with the privilege pre-check
applied before the drop -- this is the exact cell that 403'd live.

In [5]:
SG_TABLE_CAT  = f"sgtable-{uuid.uuid4().hex[:6]}"
SG_TABLE_NS   = f"ns-{SG_TABLE_CAT}"
SG_TABLE_NAME = f"sg-{uuid.uuid4().hex[:6]}"

force_delete_catalog(SG_TABLE_CAT)
create_catalog(SG_TABLE_CAT, token=tok, properties={
    "polaris.config.drop-with-purge.enabled": "true"})
create_namespace_verified(SG_TABLE_CAT, SG_TABLE_NS)

# THE FIX: check + grant CATALOG_MANAGE_CONTENT BEFORE attempting the purge
# drop -- proven live to be required even for root/service_admin.
ensure_privilege(SG_TABLE_CAT, "catalog_admin", REQUIRED_FOR_TABLE_PURGE, label="smoking-gun")

r_ct = requests.post(f"{BASE_CAT}/{SG_TABLE_CAT}/namespaces/{SG_TABLE_NS}/tables", headers=h(tok), json={
    "name": SG_TABLE_NAME,
    "location": f"s3a://{BUCKET}/{SG_TABLE_CAT}/{SG_TABLE_NS}/{SG_TABLE_NAME}/",
    "schema": {"type": "struct", "fields": [
        {"id": 1, "name": "id", "type": "long", "required": True},
        {"id": 2, "name": "name", "type": "string", "required": False}]},
    "properties": {"write.format.default": "parquet"}})
print_response(r_ct, "Create table (smoking gun)")

sg_snapshot_id, sg_written_keys, sg_commit_status = write_real_table_data(
    SG_TABLE_CAT, SG_TABLE_NS, SG_TABLE_NAME)
print(f"snapshot commit status: {sg_commit_status}, snapshot_id: {sg_snapshot_id}")
print(f"files written: {sg_written_keys}")

sg_files_before = mc.list(f"{SG_TABLE_CAT}/{SG_TABLE_NS}/{SG_TABLE_NAME}/")
print(f"\nMinIO files BEFORE purge ({len(sg_files_before)}): {sg_files_before}")

[smoking-gun] privilege check: catalog_admin@sgtable-9ee872 has {'CATALOG_MANAGE_METADATA', 'CATALOG_MANAGE_ACCESS'} -- missing ['CATALOG_MANAGE_CONTENT'] -> granting CATALOG_MANAGE_CONTENT

Create table (smoking gun):
  Status:     200
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000047133
  Body: {
  "metadata-location": "s3a://data-catalog-bucket/sgtable-9ee872/ns-sgtable-9ee872/sg-9db84c/metadata/00000-d8e195ea-51f4-4b3a-8017-d299c2458b1f.metadata.json",
  "metadata": {
    "format-version": 2,
    "table-uuid": "218d8515-45ef-4554-b564-133b009f1809",
    "location": "s3a://data-catalog-bucket/sgtable-9ee872/ns-sgtable-9ee872/sg-9db84c",
    "last-sequence-number": 0,
    "last-updated-ms": 1783315908295,
    "last-column-id": 2,
    "current-schema-id": 0,
    "schemas": [
      {
        "type": "struct",
        "schema-id": 0,
        "fields": [
          {
            "id": 1,
            "name": "id",
            "required": true,
            "type": "long"
 

In [6]:
r_drop = requests.delete(
    f"{BASE_CAT}/{SG_TABLE_CAT}/namespaces/{SG_TABLE_NS}/tables/{SG_TABLE_NAME}?purgeRequested=true",
    headers=h(tok))
print_response(r_drop, "DROP TABLE ?purgeRequested=true")

table_gone = requests.get(
    f"{BASE_CAT}/{SG_TABLE_CAT}/namespaces/{SG_TABLE_NS}/tables/{SG_TABLE_NAME}", headers=h(tok)
).status_code == 404

time.sleep(3)
sg_files_after = mc.list(f"{SG_TABLE_CAT}/{SG_TABLE_NS}/{SG_TABLE_NAME}/")

print(f"\ncatalog record gone: {table_gone}")
print(f"MinIO files AFTER purge ({len(sg_files_after)}): {sg_files_after}")
print()
if r_drop.status_code == 403:
    print("STILL 403 -- if this fires, the privilege check above did not actually")
    print("resolve it; inspect the error body and re-check REQUIRED_FOR_TABLE_PURGE.")
    print(f"  body: {r_drop.text[:300]}")
elif r_drop.status_code in (200, 204) and table_gone and len(sg_files_after) >= len(sg_files_before) and sg_files_before:
    print("CONFIRMED (this run): DROP returns success and the catalog record is")
    print("gone, but every data/manifest/metadata file written above is STILL in")
    print("MinIO. purgeRequested=true did not remove a single byte of storage.")
elif not sg_files_before:
    print("INCONCLUSIVE: no files were confirmed written before the drop -- check")
    print("the snapshot commit status in the previous cell.")
else:
    print(f"UNEXPECTED: before={len(sg_files_before)} after={len(sg_files_after)} "
          f"drop_status={r_drop.status_code} gone={table_gone} -- inspect above.")


DROP TABLE ?purgeRequested=true:
  Status:     204
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000047136
  Body: 

catalog record gone: True
MinIO files AFTER purge (6): ['sgtable-9ee872/ns-sgtable-9ee872/sg-9db84c/data/c04ceb44da6d44e1b3d5aa0e835e8b52.parquet', 'sgtable-9ee872/ns-sgtable-9ee872/sg-9db84c/metadata/00000-d8e195ea-51f4-4b3a-8017-d299c2458b1f.metadata.json', 'sgtable-9ee872/ns-sgtable-9ee872/sg-9db84c/metadata/00001-05b66690-8dac-426f-b1e1-15a7ee68d534.metadata.json', 'sgtable-9ee872/ns-sgtable-9ee872/sg-9db84c/metadata/00002-1f9c2b70-1e07-429f-9f1b-2cfe72cf31d1.metadata.json', 'sgtable-9ee872/ns-sgtable-9ee872/sg-9db84c/metadata/380eb03a8fa04eed99011267bed09f37-m0.avro', 'sgtable-9ee872/ns-sgtable-9ee872/sg-9db84c/metadata/snap-1783315908620-08f2a9533d884a52888040a370dd61d3.avro']

CONFIRMED (this run): DROP returns success and the catalog record is
gone, but every data/manifest/metadata file written above is STILL in
MinIO. purgeRequested=true did not 

In [7]:
# Deletion-ATTEMPT evidence + explicit privilege-403 vs config-403 vs
# cleanup-task-attempt classification (proven log lines from
# doc-purge-troubleshooting.md, plus the newly observed auth message).
def deletion_attempt_evidence(entity_name, minutes=20, size=300):
    hits = search_logs_raw({
        "query": {"bool": {"must": [
            {"range": {"@timestamp": {"gte": f"now-{minutes}m"}}},
            {"term": {"kubernetes.container_name.keyword": POLARIS_CONTAINER}},
            {"match_phrase": {"message": entity_name}}]},
        }, "sort": [{"@timestamp": {"order": "asc"}}]}, size=size)
    lines = [(hh["_source"].get("level",""), hh["_source"].get("loggerName",""),
              hh["_source"].get("message","")) for hh in hits]
    attempt_toks = ("scheduled cleanup task", "handling task entity id",
                    "table metadata cleanup task", "failed to handle task")
    attempt_lines = [l for l in lines if any(t in l[2].lower() for t in attempt_toks)]
    root_cause_lines = [l for l in lines if "allowedwritelocations" in l[2].lower()
                        or "allowedwritelocations" in l[1].lower()]
    return lines, attempt_lines, root_cause_lines

def classify_403(text):
    if "unable to purge entity" in text.lower():
        return "CONFIG gate (drop-with-purge.enabled)"
    if "is not authorized" in text.lower():
        return "PRIVILEGE gate (missing grant, e.g. CATALOG_MANAGE_CONTENT)"
    return "unknown"

if r_drop.status_code == 403:
    print(f"403 classification: {classify_403(r_drop.text)}")

named, attempted, root_cause = deletion_attempt_evidence(SG_TABLE_NAME)
print(f"{len(named)} log lines name this table; {len(attempted)} look like a cleanup-task attempt; "
      f"{len(root_cause)} show the documented empty-write-location root cause.")
for lvl, lg, msg in attempted[:8]:
    print(f"  [{lvl}] {lg[-30:]}: {msg[:110]}")
for lvl, lg, msg in root_cause[:4]:
    print(f"  [{lvl}] {lg[-30:]}: {msg[:110]}")
print()
if attempted:
    print("ATTEMPT CONFIRMED: Polaris DID try to clean up the table's storage --")
    print("this is not a silent no-op. Combined with the file check above, the")
    print("attempt DEMONSTRABLY FAILED (if files are still present).")
elif r_drop.status_code == 403:
    print("No cleanup-task attempt logged -- expected, since the DROP itself was")
    print("rejected before Polaris ever got to scheduling cleanup.")
else:
    print("No cleanup-task log lines matched -- either log index lag, or this")
    print("build's async task naming differs from the traced root-cause doc.")

88 log lines name this table; 2 look like a cleanup-task attempt; 8 show the documented empty-write-location root cause.
  [INFO] catalog.iceberg.IcebergCatalog: Scheduled cleanup task 4726827359302215353 for table ns-sgtable-9ee872.sg-9db84c
  [INFO] e.task.TableCleanupTaskHandler: tableIdentifier=ns-sgtable-9ee872.sg-9db84c metadataLocation=s3a://data-catalog-bucket/sgtable-9ee872/ns-sgtab
  [DEBUG] e.cache.StorageCredentialCache: key=StorageCredentialCacheKey{realmId=POLARIS, catalogId=0, storageConfigSerializedStr={"@type":"AwsStorageCon
  [DEBUG] e.cache.StorageCredentialCache: key=StorageCredentialCacheKey{realmId=POLARIS, catalogId=0, storageConfigSerializedStr={"@type":"AwsStorageCon
  [DEBUG] e.cache.StorageCredentialCache: key=StorageCredentialCacheKey{realmId=POLARIS, catalogId=0, storageConfigSerializedStr={"@type":"AwsStorageCon
  [DEBUG] e.cache.StorageCredentialCache: key=StorageCredentialCacheKey{realmId=POLARIS, catalogId=0, storageConfigSerializedStr={"@type":"AwsStor

## Part 2 — Config-gate matrix (table only), `purgeRequested` EXPLICITLY controlled

Every case below now sends an explicit `purgeRequested` value (never left
absent) so the purge path -- and therefore the config gate -- actually
engages. `ensure_privilege()` runs first, IF `grant=True`; when `grant=False`
we deliberately withhold `CATALOG_MANAGE_CONTENT` to observe the privilege
gate distinctly from the config gate (see `classify_403`). This is the fix
for the live run's C1/C2 "unexpected 204" (purge was never actually being
requested) and the smoking gun's "unexpected 403" (privilege was never
checked).

In [8]:
results = {}

def _ns_for(cat):
    return f"ns-{cat}"

def make_table(cat, name):
    ns = _ns_for(cat)
    url = f"{BASE_CAT}/{cat}/namespaces/{ns}/tables"
    body = {"name": name, "location": f"s3a://{BUCKET}/{cat}/{ns}/{name}/",
             "schema": {"type": "struct", "fields": [
                 {"id": 1, "name": "id", "type": "long", "required": True}]},
             "properties": {"write.format.default": "parquet"}}
    r = requests.post(url, headers=h(tok), json=body)
    if r.status_code in (200, 201):
        return r
    if r.status_code == 500 and ("metadata" in r.text or "NullPointer" in r.text):
        time.sleep(0.6)
        class _OK:
            status_code = 200
            text = '{"note":"table committed; 500 was a read-after-write lag (false failure)"}'
        return _OK()
    return r

def cat_files(cat, name):
    return mc.list(f"{cat}/{_ns_for(cat)}/{name}/")

def fresh_catalog_for_case(case, drop_purge, grant):
    cat = f"gm-t-{case.lower()}-{uuid.uuid4().hex[:5]}"
    create_catalog(cat, token=tok, properties={
        "polaris.config.drop-with-purge.enabled": str(drop_purge).lower()})
    cfg = get_catalog_config(cat)
    assert cfg["drop-with-purge.enabled"] == str(drop_purge).lower(), f"{cat}: {cfg}"
    if grant:
        ensure_privilege(cat, "catalog_admin", REQUIRED_FOR_TABLE_PURGE, label=case)
    create_namespace_verified(cat, _ns_for(cat))
    return cat, cfg

def run_case(case, drop_purge, grant, purge_requested, expected_status=(200, 204)):
    """purge_requested is now REQUIRED to be explicit (True/False/None) at
    the call site -- no silent 'absent means untested' cases."""
    if isinstance(expected_status, int):
        expected_status = (expected_status,)
    key = case

    cat, cfg_before = fresh_catalog_for_case(case, drop_purge, grant)
    ename = f"{case}-{uuid.uuid4().hex[:6]}"
    cr = make_table(cat, ename)
    if cr.status_code not in (200, 201):
        results[key] = {"cat": cat, "name": ename, "drop_purge": drop_purge, "grant": grant,
                        "purge_req": {True: "true", False: "false", None: "absent"}[purge_requested],
                        "status": "CREATE_FAIL", "gone": None, "before": 0, "after": 0,
                        "err": cr.text[:200], "expected": expected_status}
        print(f"[{key}] CREATE FAILED [{cr.status_code}] -- {cr.text[:150]}")
        force_delete_catalog(cat)
        return results[key]

    files_before = cat_files(cat, ename)
    url = f"{BASE_CAT}/{cat}/namespaces/{_ns_for(cat)}/tables/{ename}"
    if purge_requested is True:
        url += "?purgeRequested=true"
    elif purge_requested is False:
        url += "?purgeRequested=false"
    t0 = time.time()
    r = requests.delete(url, headers=h(tok))
    elapsed = time.time() - t0

    exists_after = requests.get(
        f"{BASE_CAT}/{cat}/namespaces/{_ns_for(cat)}/tables/{ename}", headers=h(tok)
    ).status_code == 200
    gone = not exists_after
    time.sleep(1.5)
    files_after = cat_files(cat, ename)

    pr = {True: "true", False: "false", None: "absent"}[purge_requested]
    fo = ("orphaned" if len(files_after) >= len(files_before) and len(files_before) > 0 else
          "deleted"  if len(files_after) == 0 and len(files_before) > 0 else
          f"{len(files_before)}->{len(files_after)}")

    results[key] = {"cat": cat, "name": ename, "drop_purge": drop_purge, "grant": grant, "purge_req": pr,
                    "status": r.status_code, "gone": gone, "before": len(files_before),
                    "after": len(files_after), "file_state": fo,
                    "err": r.text[:250] if r.status_code >= 400 else "",
                    "gate": classify_403(r.text) if r.status_code == 403 else "-",
                    "expected": expected_status}

    is_error = r.status_code >= 400
    unexpected = r.status_code not in expected_status
    flag = "UNEXPECTED" if unexpected else ("warn" if is_error else "ok")
    print(f"[{key}] {flag} status={r.status_code} (expected {expected_status})  "
          f"file={fo}  gone={gone}  gate={results[key]['gate']}  {elapsed*1000:.0f}ms")
    if DEBUG or (is_error and unexpected):
        print(f"   catalog={cat}  cfg={cfg_before}  files_before={files_before}  files_after={files_after}")
        print(f"   DELETE {url}\n   response [{r.status_code}]: {r.text[:300]}")
    elif is_error:
        print(f"      (expected) {r.text[:150]}")

    force_delete_catalog(cat)
    return results[key]

print("table-only run_case ready (purgeRequested always explicit)")

table-only run_case ready (purgeRequested always explicit)


In [9]:
# P1 -- config CLOSED + no grant, purge requested -> some 403 (either gate could fire first)
run_case("P1", drop_purge=False, grant=False, purge_requested=True, expected_status=403)

[P1] warn status=403 (expected (403,))  file=orphaned  gone=False  gate=PRIVILEGE gate (missing grant, e.g. CATALOG_MANAGE_CONTENT)  7ms
      (expected) {"error":{"message":"Principal 'root' with activated PrincipalRoles '[service_admin]' and activated grants via '[service_admin, catalog_admin]' is not


{'cat': 'gm-t-p1-8b156',
 'name': 'P1-fc84db',
 'drop_purge': False,
 'grant': False,
 'purge_req': 'true',
 'status': 403,
 'gone': False,
 'before': 1,
 'after': 1,
 'file_state': 'orphaned',
 'err': '{"error":{"message":"Principal \'root\' with activated PrincipalRoles \'[service_admin]\' and activated grants via \'[service_admin, catalog_admin]\' is not authorized for op DROP_TABLE_WITH_PURGE","type":"ForbiddenException","code":403}}',
 'gate': 'PRIVILEGE gate (missing grant, e.g. CATALOG_MANAGE_CONTENT)',
 'expected': (403,)}

In [10]:
# P2 -- config CLOSED + granted, purge requested -> isolates the CONFIG gate alone
run_case("P2", drop_purge=False, grant=True, purge_requested=True, expected_status=403)

[P2] privilege check: catalog_admin@gm-t-p2-8541b has {'CATALOG_MANAGE_METADATA', 'CATALOG_MANAGE_ACCESS'} -- missing ['CATALOG_MANAGE_CONTENT'] -> granting CATALOG_MANAGE_CONTENT
[P2] warn status=403 (expected (403,))  file=orphaned  gone=False  gate=CONFIG gate (drop-with-purge.enabled)  23ms
      (expected) {"error":{"message":"Unable to purge entity: P2-6d5e09. To enable this feature, set the Polaris configuration DROP_WITH_PURGE_ENABLED or the catalog c


{'cat': 'gm-t-p2-8541b',
 'name': 'P2-6d5e09',
 'drop_purge': False,
 'grant': True,
 'purge_req': 'true',
 'status': 403,
 'gone': False,
 'before': 1,
 'after': 1,
 'file_state': 'orphaned',
 'err': '{"error":{"message":"Unable to purge entity: P2-6d5e09. To enable this feature, set the Polaris configuration DROP_WITH_PURGE_ENABLED or the catalog configuration polaris.config.drop-with-purge.enabled","type":"ForbiddenException","code":403}}',
 'gate': 'CONFIG gate (drop-with-purge.enabled)',
 'expected': (403,)}

In [11]:
# P3 -- config OPEN + NO grant, purge requested -> isolates the PRIVILEGE gate alone (the new finding)
run_case("P3", drop_purge=True, grant=False, purge_requested=True, expected_status=403)

[P3] warn status=403 (expected (403,))  file=orphaned  gone=False  gate=PRIVILEGE gate (missing grant, e.g. CATALOG_MANAGE_CONTENT)  7ms
      (expected) {"error":{"message":"Principal 'root' with activated PrincipalRoles '[service_admin]' and activated grants via '[service_admin, catalog_admin]' is not


{'cat': 'gm-t-p3-d8cd5',
 'name': 'P3-774351',
 'drop_purge': True,
 'grant': False,
 'purge_req': 'true',
 'status': 403,
 'gone': False,
 'before': 1,
 'after': 1,
 'file_state': 'orphaned',
 'err': '{"error":{"message":"Principal \'root\' with activated PrincipalRoles \'[service_admin]\' and activated grants via \'[service_admin, catalog_admin]\' is not authorized for op DROP_TABLE_WITH_PURGE","type":"ForbiddenException","code":403}}',
 'gate': 'PRIVILEGE gate (missing grant, e.g. CATALOG_MANAGE_CONTENT)',
 'expected': (403,)}

In [12]:
# P4 -- config OPEN + granted, purge requested -> everything on, expect success
run_case("P4", drop_purge=True, grant=True, purge_requested=True, expected_status=(200, 204))

[P4] privilege check: catalog_admin@gm-t-p4-1dbe0 has {'CATALOG_MANAGE_METADATA', 'CATALOG_MANAGE_ACCESS'} -- missing ['CATALOG_MANAGE_CONTENT'] -> granting CATALOG_MANAGE_CONTENT
[P4] ok status=204 (expected (200, 204))  file=orphaned  gone=True  gate=-  30ms


{'cat': 'gm-t-p4-1dbe0',
 'name': 'P4-f6ec0b',
 'drop_purge': True,
 'grant': True,
 'purge_req': 'true',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'file_state': 'orphaned',
 'err': '',
 'gate': '-',
 'expected': (200, 204)}

In [13]:
# P5 -- purge NOT requested at all, config closed, no grant -> plain drop unaffected by either gate
run_case("P5", drop_purge=False, grant=False, purge_requested=False, expected_status=(200, 204))

[P5] ok status=204 (expected (200, 204))  file=orphaned  gone=True  gate=-  13ms


{'cat': 'gm-t-p5-20438',
 'name': 'P5-689ea3',
 'drop_purge': False,
 'grant': False,
 'purge_req': 'false',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'file_state': 'orphaned',
 'err': '',
 'gate': '-',
 'expected': (200, 204)}

In [14]:
# P6 -- idempotency: drop an already-dropped table (expect 404 on the 2nd drop)
cat, _ = fresh_catalog_for_case("P6", drop_purge=True, grant=True)
ename = f"P6-{uuid.uuid4().hex[:6]}"
make_table(cat, ename)
url = f"{BASE_CAT}/{cat}/namespaces/{_ns_for(cat)}/tables/{ename}?purgeRequested=true"
r1 = requests.delete(url, headers=h(tok))
r2 = requests.delete(url, headers=h(tok))
results["P6"] = {"cat": cat, "name": ename, "drop_purge": True, "grant": True, "purge_req": "true",
                 "status": r2.status_code, "gone": True, "before": 0, "after": 0, "file_state": "-",
                 "gate": "-", "err": r2.text[:150] if r2.status_code >= 400 else "", "expected": (404,)}
print(f"[P6] 1st drop={r1.status_code}  2nd drop={r2.status_code} (expected 404)")
force_delete_catalog(cat)

[P6] privilege check: catalog_admin@gm-t-p6-e3b2a has {'CATALOG_MANAGE_METADATA', 'CATALOG_MANAGE_ACCESS'} -- missing ['CATALOG_MANAGE_CONTENT'] -> granting CATALOG_MANAGE_CONTENT
[P6] 1st drop=204  2nd drop=404 (expected 404)


True

## Part 3 — Privilege isolation: non-root, isolating `TABLE_DROP` vs `CATALOG_MANAGE_CONTENT`

Reused as-is from the combined notebook -- this part already worked correctly
live and produced the real finding driving the fix above (`TABLE_DROP` alone
insufficient; `CATALOG_MANAGE_CONTENT` sufficient). Kept here so this
notebook is self-contained.

In [15]:
def nonroot_drop_probe(cat, ename, privileges, label):
    P  = f"nr-t-prin-{uuid.uuid4().hex[:6]}"
    PR = f"nr-t-prole-{uuid.uuid4().hex[:6]}"
    CR = f"nr-t-crole-{uuid.uuid4().hex[:6]}"

    rp = create_principal(P, token=tok)
    creds = rp.json()
    client_id     = creds.get("credentials", {}).get("clientId") or creds.get("clientId")
    client_secret = creds.get("credentials", {}).get("clientSecret") or creds.get("clientSecret")

    create_principal_role(PR, token=tok)
    create_catalog_role(catalog=cat, name=CR, token=tok)
    assign_catalog_role_to_principal_role(catalog=cat, pr=PR, cr=CR, token=tok)
    assign_principal_role_to_principal(principal=P, pr=PR, token=tok)
    for priv in privileges:
        grant_privilege(catalog=cat, cr=CR, privilege=priv, token=tok)

    tr = get_token(client_id=client_id, client_secret=client_secret)
    ltok = tr.json().get("access_token") if tr.status_code == 200 else None
    result = {"label": label, "privileges": list(privileges), "status": None, "err": ""}
    if ltok:
        rh = {"Authorization": f"Bearer {ltok}", "Polaris-Realm": REALM, "Content-Type": "application/json"}
        url = f"{BASE_CAT}/{cat}/namespaces/{_ns_for(cat)}/tables/{ename}?purgeRequested=true"
        r = requests.delete(url, headers=rh)
        result["status"] = r.status_code
        result["err"] = r.text[:250] if r.status_code >= 400 else ""
        print(f"[{label}] non-root DROP (privileges={privileges}) -> {r.status_code}"
              + (f"  {r.text[:160]}" if r.status_code >= 400 else "  ok"))
    else:
        print(f"[{label}] could not obtain a token -- token request status {tr.status_code}")

    requests.delete(f"{BASE_MGMT}/principals/{P}", headers=h(tok))
    requests.delete(f"{BASE_MGMT}/principal-roles/{PR}", headers=h(tok))
    requests.delete(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{CR}", headers=h(tok))
    return result

priv_results = {}
print("nonroot_drop_probe ready (table)")

nonroot_drop_probe ready (table)


In [16]:
# TABLE -- baseline (no drop-authorizing privilege) -> expect 403
cat, _ = fresh_catalog_for_case("PR-base", drop_purge=True, grant=False)
ename = f"prb-{uuid.uuid4().hex[:6]}"
make_table(cat, ename)
priv_results["table_baseline"] = nonroot_drop_probe(cat, ename,
    ["TABLE_LIST", "TABLE_CREATE", "VIEW_LIST"], "TABLE baseline (no drop priv)")
force_delete_catalog(cat)

[TABLE baseline (no drop priv)] non-root DROP (privileges=['TABLE_LIST', 'TABLE_CREATE', 'VIEW_LIST']) -> 403  {"error":{"message":"Principal 'nr-t-prin-bfeeee' with activated PrincipalRoles '[nr-t-prole-060a16]' and activated grants via '[nr-t-crole-2721dc, nr-t-prole-0


True

In [17]:
# TABLE -- baseline + TABLE_DROP ONLY -- PROVEN LIVE to still be 403
# (contradicts doc-privilege-results.md / doc-privilege-matrix-plan.md, which
# claim TABLE_DROP alone is sufficient -- it is NOT, for the purge path).
cat, _ = fresh_catalog_for_case("PR-drop", drop_purge=True, grant=False)
ename = f"prd-{uuid.uuid4().hex[:6]}"
make_table(cat, ename)
priv_results["table_with_table_drop"] = nonroot_drop_probe(cat, ename,
    ["TABLE_LIST", "TABLE_CREATE", "TABLE_DROP"], "TABLE + TABLE_DROP only")
force_delete_catalog(cat)

[TABLE + TABLE_DROP only] non-root DROP (privileges=['TABLE_LIST', 'TABLE_CREATE', 'TABLE_DROP']) -> 403  {"error":{"message":"Principal 'nr-t-prin-1ded2a' with activated PrincipalRoles '[nr-t-prole-a56706]' and activated grants via '[nr-t-crole-bab459, nr-t-prole-a


True

In [18]:
# TABLE -- baseline + CATALOG_MANAGE_CONTENT -- PROVEN LIVE to be sufficient.
cat, _ = fresh_catalog_for_case("PR-cmc", drop_purge=True, grant=False)
ename = f"prc-{uuid.uuid4().hex[:6]}"
make_table(cat, ename)
priv_results["table_with_cmc"] = nonroot_drop_probe(cat, ename,
    ["TABLE_LIST", "TABLE_CREATE", "CATALOG_MANAGE_CONTENT"], "TABLE + CATALOG_MANAGE_CONTENT")
force_delete_catalog(cat)

[TABLE + CATALOG_MANAGE_CONTENT] non-root DROP (privileges=['TABLE_LIST', 'TABLE_CREATE', 'CATALOG_MANAGE_CONTENT']) -> 204  ok


True

In [19]:
print("="*78)
print("PRIVILEGE ISOLATION -- summary (table)")
print("="*78)
for k, v in priv_results.items():
    print(f"  {k:24s} status={v['status']!s:5} privileges={v['privileges']}")
print()

tb  = priv_results.get("table_baseline", {})
ttd = priv_results.get("table_with_table_drop", {})
tcm = priv_results.get("table_with_cmc", {})
if tb.get("status") == 403:
    print("baseline (no drop grant) is 403 -- privilege gate confirmed for tables")
    print("(unlike views, where root/service_admin bypasses this check entirely).")
    if ttd.get("status") in (200, 204):
        print("  -> TABLE_DROP ALONE is sufficient.")
    elif ttd.get("status") == 403:
        print("  -> TABLE_DROP ALONE is NOT sufficient -- corrects")
        print("     doc-privilege-results.md / doc-privilege-matrix-plan.md.")
    if tcm.get("status") in (200, 204):
        print("  -> CATALOG_MANAGE_CONTENT IS sufficient -- this is the privilege")
        print("     REQUIRED_FOR_TABLE_PURGE (used by ensure_privilege above) should grant.")
elif tb.get("status") in (200, 204):
    print("baseline succeeded with NO drop-authorizing privilege -- the privilege")
    print("gate did not fire for tables in this run; investigate baseline grants.")

PRIVILEGE ISOLATION -- summary (table)
  table_baseline           status=403   privileges=['TABLE_LIST', 'TABLE_CREATE', 'VIEW_LIST']
  table_with_table_drop    status=403   privileges=['TABLE_LIST', 'TABLE_CREATE', 'TABLE_DROP']
  table_with_cmc           status=204   privileges=['TABLE_LIST', 'TABLE_CREATE', 'CATALOG_MANAGE_CONTENT']

baseline (no drop grant) is 403 -- privilege gate confirmed for tables
(unlike views, where root/service_admin bypasses this check entirely).
  -> TABLE_DROP ALONE is NOT sufficient -- corrects
     doc-privilege-results.md / doc-privilege-matrix-plan.md.
  -> CATALOG_MANAGE_CONTENT IS sufficient -- this is the privilege
     REQUIRED_FOR_TABLE_PURGE (used by ensure_privilege above) should grant.


## Truth table -- table-only case matrix

In [20]:
print("="*100)
print(f"TABLE purge -- case matrix (Polaris {POLARIS_VERSION})")
print("="*100)
print(f"{'case':6}{'drop-purge':11}{'grant':7}{'purgeReq':9}{'status':8}{'gate':42}{'file':10}{'gone'}")
print("-"*110)
for case in ("P1", "P2", "P3", "P4", "P5", "P6"):
    r = results.get(case)
    if not r:
        print(f"{case:6}(not run)"); continue
    print(f"{case:6}{str(r['drop_purge']):11}{str(r['grant']):7}{r['purge_req']:9}"
          f"{str(r['status']):8}{r.get('gate','-'):42}{r.get('file_state','-'):10}{r['gone']}")
    if r.get("err"):
        print(f"        -> {r['err'][:100]}")

print()
print("="*100)
print("VERDICT")
print("="*100)
print("Smoking-gun (Part 1, with the privilege fix applied): see the conclusion")
print("printed under the drop cell -- table purge attempts cleanup, fails on the")
print("empty-write-location bug (#379), files orphan.")
print()
print("Config matrix (Part 2): P1-P3 isolate CONFIG-403 vs PRIVILEGE-403 by")
print("message text (see the 'gate' column) -- P2 should be CONFIG gate, P3 should")
print("be PRIVILEGE gate, distinguishing the two mechanisms that were previously")
print("conflated in the combined notebook's unexpected results.")
print()
print("Privilege isolation (Part 3): TABLE_DROP alone vs CATALOG_MANAGE_CONTENT --")
print("see the summary above for which one actually authorizes the purge.")

TABLE purge -- case matrix (Polaris 1.3.0)
case  drop-purge grant  purgeReq status  gate                                      file      gone
--------------------------------------------------------------------------------------------------------------
P1    False      False  true     403     PRIVILEGE gate (missing grant, e.g. CATALOG_MANAGE_CONTENT)orphaned  False
        -> {"error":{"message":"Principal 'root' with activated PrincipalRoles '[service_admin]' and activated 
P2    False      True   true     403     CONFIG gate (drop-with-purge.enabled)     orphaned  False
        -> {"error":{"message":"Unable to purge entity: P2-6d5e09. To enable this feature, set the Polaris conf
P3    True       False  true     403     PRIVILEGE gate (missing grant, e.g. CATALOG_MANAGE_CONTENT)orphaned  False
        -> {"error":{"message":"Principal 'root' with activated PrincipalRoles '[service_admin]' and activated 
P4    True       True   true     204     -                                       

## Cleanup

Sweeps every catalog this notebook created. Same "no persistent pool to
dispose" note as the combined notebook applies here (only `requests` HTTP
calls + a per-call OpenSearch client).

In [21]:
all_cats = [c["name"] for c in requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])]
leftover = [c for c in all_cats if c.startswith(("sgtable-", "gm-t-")) or c.startswith("nr-t-")]
print(f"leftover catalogs from this notebook: {leftover}")
for c in leftover:
    force_delete_catalog(c)

del results, priv_results
import gc; gc.collect()
print("cleanup complete")

leftover catalogs from this notebook: ['sgtable-9ee872']
cleanup complete


## Summary — table purge privilege model (fill in after a live run)

Expected conclusion, pending live confirmation:
1. Table purge (`purgeRequested=true`) is gated by a DISTINCT authorization op
   (`DROP_TABLE_WITH_PURGE`) that even root/`service_admin` does not bypass --
   this is DIFFERENT from view drops, where root does bypass. This is the
   "table's privilege is different from view" the request called out.
2. `CATALOG_MANAGE_CONTENT` authorizes it; `TABLE_DROP` alone does not (this
   corrects `doc-privilege-results.md` / `doc-privilege-matrix-plan.md`).
3. The underlying storage question is unchanged: once authorized, the purge
   still only *attempts* cleanup and fails (#379) -- authorization and actual
   file deletion are two separate gates, both broken in different ways.

Per `CLAUDE.md` DoD: once run clean (Restart & Run All, zero errors), record
the real per-case statuses here and in `MEMORY.md`, and correct
`doc-privilege-results.md` §"Table DROP" to note `TABLE_DROP` alone is
insufficient for a *purging* drop (still sufficient for a plain drop).